In [ ]:
# %% [markdown]
# ## 0) Setup & Imports

# %%
import os, re, json
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils import shuffle
import joblib

In [ ]:
# %% [markdown]
# ## 1) Paths & Reproducibility

# %%
SEED = 42
np.random.seed(SEED)

# Detect project root (assumes this notebook lives under <repo>/notebooks)
NB_DIR = Path.cwd()
if NB_DIR.name != "notebooks":
    # if user ran from repo root, emulate notebooks/ as CWD for consistency
    NB_DIR = Path.cwd() / "notebooks"

ROOT = NB_DIR.parent
DATA_RAW = ROOT / "data" / "raw"
DATA_PRE = ROOT / "data" / "preprocessed"

REDDIT_DIR = DATA_RAW / "reddit"
TWITTER_FILE = DATA_RAW / "Mental-Health-Twitter.csv"

# Create output folders
(DATA_PRE / "experiments" / "expA_red80_tw20").mkdir(parents=True, exist_ok=True)
(DATA_PRE / "experiments" / "expB_tw80_red20").mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Reddit dir exists:", REDDIT_DIR.exists())
print("Twitter file exists:", TWITTER_FILE.exists())
print("Reddit files:", sorted([p.name for p in REDDIT_DIR.glob("*.csv")])[:5], "...")


In [ ]:
# %% [markdown]
# ## 2) Label Policy (Binary)
# Mental-health (1): Addiction, ADHD, Alcoholism, Anxiety, MentalHealth, SuicideWatch
# Control (0):      Conspiracy, Fitness, Jokes, Meditation, Relationships, Teaching

# %%
MENTAL_SUBS = {
    "addiction", "adhd", "alcoholism", "anxiety", "mentalhealth", "suicidewatch"
}
CONTROL_SUBS = {
    "conspiracy", "fitness", "jokes", "meditation", "relationships", "teaching", "relationship"  # some dumps use singular
}


In [ ]:
# %% [markdown]
# ## 3) Cleaning tailored for Random Forest
# - Remove URLs, non-letters, collapse whitespace, lowercase
# - No stemming/lemmatization (trees handle forms fine)
# - We’ll use compact TF-IDF later (2–5k features, unigrams+bigrams)

# %%
def clean_for_rf(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


In [ ]:
# %% [markdown]
# ## 4) Load Reddit (12 subreddits → binary labels)
# Expects: files like `anxiety_2019_features_tfidf_256.csv` with a `post` column

# %%
def load_reddit_binary(reddit_dir: Path) -> pd.DataFrame:
    rows = []
    for csv in sorted(reddit_dir.glob("*.csv")):
        name = csv.stem.split("_")[0].lower()  # e.g., 'anxiety'
        df = pd.read_csv(csv)
        if "post" not in df.columns:
            raise ValueError(f"'post' column not found in {csv.name}")
        if name in MENTAL_SUBS:
            label = 1
        elif name in CONTROL_SUBS:
            label = 0
        else:
            # If an unexpected file appears, default to control but log it
            print(f"[WARN] Unrecognized subreddit '{name}' → defaulting label=0")
            label = 0
        rows.append(pd.DataFrame({
            "text": df["post"].astype(str),
            "label": label,
            "domain": "reddit",
            "source": name
        }))
    out = pd.concat(rows, ignore_index=True)
    out["text"] = out["text"].map(clean_for_rf)
    return shuffle(out, random_state=SEED)

reddit_df = load_reddit_binary(REDDIT_DIR)
print(reddit_df.shape)
print(reddit_df["label"].value_counts())
print(reddit_df["source"].value_counts().head(12))


In [ ]:
# %% [markdown]
# ## 5) Load Twitter (binary)
# Expects columns: `post_text` and `label` (0/1). If names differ, adapt below.

# %%
tw = pd.read_csv(TWITTER_FILE)
# Normalize column names if needed
if "tweet" in tw.columns and "label" in tw.columns:
    tw = tw.rename(columns={"tweet": "post_text"})
if "post" in tw.columns and "label" in tw.columns:
    tw = tw.rename(columns={"post": "post_text"})

assert "post_text" in tw.columns and "label" in tw.columns, "Twitter CSV must have post_text and label columns."

twitter_df = pd.DataFrame({
    "text": tw["post_text"].astype(str).map(clean_for_rf),
    "label": tw["label"].astype(int),
    "domain": "twitter",
    "source": "twitter"
})
twitter_df = shuffle(twitter_df, random_state=SEED)

print(twitter_df.shape)
print(twitter_df["label"].value_counts())


In [ ]:
# %% [markdown]
# ## 6) Per-domain base splits (70/15/15, stratified)

# %%
def stratified_70_15_15(df: pd.DataFrame, label_col="label"):
    train, temp = train_test_split(df, test_size=0.30, stratify=df[label_col], random_state=SEED)
    val, test = train_test_split(temp, test_size=0.50, stratify=temp[label_col], random_state=SEED)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

red_train, red_val, red_test = stratified_70_15_15(reddit_df)
tw_train,  tw_val,  tw_test  = stratified_70_15_15(twitter_df)

for name, part in [("Reddit-train", red_train), ("Reddit-val", red_val), ("Reddit-test", red_test),
                   ("Twitter-train", tw_train), ("Twitter-val", tw_val), ("Twitter-test", tw_test)]:
    print(name, part["label"].value_counts(normalize=True).round(3).to_dict(), len(part))


In [ ]:
# %% [markdown]
# ## 7) Build experiment mixes (80/20)
# - Exp-A (Red80/Tw20): train/val drawn as 80% Reddit + 20% Twitter; Test = **full Twitter test**
# - Exp-B (Tw80/Red20): train/val drawn as 80% Twitter + 20% Reddit; Test = **full Reddit test**

# %%
def mix_datasets(primary_train, primary_val, secondary_train, secondary_val, mix_ratio=0.20):
    # sample from secondary to be `mix_ratio` of the final size relative to primary counts
    # Keep class balance by stratified sampling within secondary sets
    def stratified_fraction(df, frac):
        if frac <= 0:
            return df.iloc[0:0].copy()
        out = (df.groupby("label", group_keys=False)
                 .apply(lambda g: g.sample(frac=min(frac, 1.0), random_state=SEED)))
        return out.reset_index(drop=True)

    # Compute fractions such that secondary ≈ mix_ratio of final (primary + secondary)
    # Solve: sec = mix_ratio * (prim + sec) -> sec = mix_ratio/(1-mix_ratio) * prim
    # Implement as fraction of secondary needed relative to its size, capped at 1.0
    ratio = mix_ratio / (1 - mix_ratio)
    need_train = min(1.0, ratio * (len(primary_train) / max(len(secondary_train), 1)))
    need_val   = min(1.0, ratio * (len(primary_val)   / max(len(secondary_val),   1)))

    sec_train_sub = stratified_fraction(secondary_train, need_train)
    sec_val_sub   = stratified_fraction(secondary_val,   need_val)

    train_mix = shuffle(pd.concat([primary_train, sec_train_sub], ignore_index=True), random_state=SEED)
    val_mix   = shuffle(pd.concat([primary_val,   sec_val_sub],   ignore_index=True), random_state=SEED)
    return train_mix, val_mix

# Exp-A: primary=Reddit, secondary=Twitter
expA_train, expA_val = mix_datasets(red_train, red_val, tw_train, tw_val, mix_ratio=0.20)
expA_test = tw_test.copy()   # evaluate on the *other* full domain

# Exp-B: primary=Twitter, secondary=Reddit
expB_train, expB_val = mix_datasets(tw_train, tw_val, red_train, red_val, mix_ratio=0.20)
expB_test = red_test.copy()

for tag, d in [("expA_train", expA_train), ("expA_val", expA_val), ("expA_test", expA_test),
               ("expB_train", expB_train), ("expB_val", expB_val), ("expB_test", expB_test)]:
    print(tag, "size:", len(d), "| domain mix:", d["domain"].value_counts(normalize=True).round(2).to_dict(),
          "| class bal:", d["label"].value_counts(normalize=True).round(2).to_dict())


In [ ]:
# %% [markdown]
# ## 8) TF-IDF for Random Forest (fit on training mix per experiment)
# - Compact vocabulary: 3k features, unigrams+bigrams, min_df=3, max_df=0.9, sublinear_tf=True
# - Fit vectorizer **only** on the training split of each experiment
# - Transform val/test with the same vectorizer

# %%
VEC_KW = dict(
    max_features=3000,
    min_df=3,
    max_df=0.90,
    ngram_range=(1,2),
    sublinear_tf=True
)

def vectorize_and_save(exp_name, train_df, val_df, test_df):
    out_dir = DATA_PRE / "experiments" / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)

    vec = TfidfVectorizer(**VEC_KW)
    Xtr = vec.fit_transform(train_df["text"])
    Xva = vec.transform(val_df["text"])
    Xte = vec.transform(test_df["text"])

    # Save vectorizer
    joblib.dump(vec, out_dir / "tfidf_rf.pkl")

    # Save numpy CSR matrices as compressed npz + labels in csv for convenience
    # (CSV of dense arrays can be huge; we keep labels separately and matrices as .npz)
    from scipy import sparse
    sparse.save_npz(out_dir / "X_train.npz", Xtr)
    sparse.save_npz(out_dir / "X_val.npz",   Xva)
    sparse.save_npz(out_dir / "X_test.npz",  Xte)

    train_df[["label","domain","source"]].to_csv(out_dir / "y_train.csv", index=False)
    val_df[["label","domain","source"]].to_csv(out_dir / "y_val.csv", index=False)
    test_df[["label","domain","source"]].to_csv(out_dir / "y_test.csv", index=False)

    # Small metadata for sanity checks
    meta = {
        "exp_name": exp_name,
        "vec_params": VEC_KW,
        "sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
        "class_balance_train": train_df["label"].value_counts(normalize=True).to_dict(),
        "domain_mix_train": train_df["domain"].value_counts(normalize=True).to_dict(),
    }
    with open(out_dir / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved {exp_name} to:", out_dir)

# Build & save both experiments
vectorize_and_save("expA_red80_tw20", expA_train, expA_val, expA_test)
vectorize_and_save("expB_tw80_red20", expB_train, expB_val, expB_test)


In [ ]:
# %% [markdown]
# ## 9) What you’ll use in the training notebooks
# - Load tfidf_rf.pkl, X_train.npz, X_val.npz, X_test.npz, and y_*.csv from a chosen experiment folder.
# - Train RandomForest on X_train / y_train, tune on X_val, report on X_test.

# %%
print("Exp-A files:", sorted(os.listdir(DATA_PRE / "experiments" / "expA_red80_tw20")))
print("Exp-B files:", sorted(os.listdir(DATA_PRE / "experiments" / "expB_tw80_red20")))
